In [ ]:
from torch import nn, optim, torch
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np
import sys
import os
sys.path.append(os.path.abspath(".."))

%load_ext autoreload
%autoreload 2
from src.features import pull_features_per_subject


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
# clean data
df = pd.read_csv("../data/intermediate/pamap_cleaned_df.csv")
X, y, subjects = pull_features_per_subject(df, nn=True) # nn = True to extract flattened raw data for neural network input

In [ ]:
print(X.shape)  # Should be (sample_idx, num_samples, num_features)

(689, 500, 39)


## Set Up CNN and Dataset Objects

In [21]:
class PAMAPDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [22]:
class ActivityCNN(nn.Module):
    def __init__(self, n_channels, n_classes):
        super().__init__()

        self.conv1 = nn.Conv1d(in_channels=n_channels, out_channels=64, kernel_size=7, padding=3)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(128, n_classes)
    
    def forward(self, x):
        x = x.transpose(1, 2)  # Reshape to (batch_size, n_channels, sequence_length)
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = self.pool(x)
        x = x.squeeze(-1)
        return self.fc(x)

In [23]:
CNN_model = ActivityCNN(n_channels=39, n_classes=12)

In [24]:
# encode labels
le = LabelEncoder()
y_enc = le.fit_transform(y)
num_classes = len(le.classes_)

In [25]:
# perform train-val split on baseline model
X_train, X_val, y_train, y_val = train_test_split(
    X, y_enc,
    test_size=0.2,
    random_state=42,
    stratify=y_enc
)

In [26]:
train_loader = DataLoader(
    PAMAPDataset(X_train, y_train),
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    PAMAPDataset(X_val, y_val),
    batch_size=32
)

In [29]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = ActivityCNN(n_channels=39, n_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
for epoch in range(10):

    model.train()
    train_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # ---- validation ----
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            preds = model(X_batch).argmax(dim=1)

            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

    acc = correct / total

    print(f"Epoch {epoch+1} | Loss: {train_loss:.3f} | Val Acc: {acc:.3f}")

Epoch 1 | Loss: 31.280 | Val Acc: 0.601
Epoch 2 | Loss: 16.119 | Val Acc: 0.804
Epoch 3 | Loss: 9.897 | Val Acc: 0.826
Epoch 4 | Loss: 7.176 | Val Acc: 0.848
Epoch 5 | Loss: 5.692 | Val Acc: 0.841
Epoch 6 | Loss: 4.078 | Val Acc: 0.906
Epoch 7 | Loss: 5.505 | Val Acc: 0.884
Epoch 8 | Loss: 3.717 | Val Acc: 0.884
Epoch 9 | Loss: 3.248 | Val Acc: 0.862
Epoch 10 | Loss: 3.006 | Val Acc: 0.928


### CNN + LSTM

In [ ]:
class CNNLSTM(nn.Module):
    def __init__(self, n_channels, n_classes, hidden_size=128, lstm_layers=1):
        super().__init__()
        self.conv1 = nn.Conv1d(n_channels, 64, kernel_size=7, padding=3)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=5,padding=2)
        self.pool = nn.MaxPool1d(kernel_size=2, stride=2)
        self.lstm = nn.LSTM(input_size=128, hidden_size=hidden_size, num_layers=lstm_layers, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_size, n_classes)

    def forward(self, x):
        # x = (batch, 500, 39)
        x = x.transpose(1, 2)
        # (batch, 39, 500)
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        # (batch, 128, reduced_time)
        x = x.transpose(1, 2)
        # (batch, reduced_time, 128)
        lstm_out, (h_n, c_n) = self.lstm(x)
        x = h_n[-1]
        x = self.droupout(x)
        return self.fc(x)